In [7]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from diffusers import AutoencoderDC
import numpy as np

In [5]:
class ConcatenateImgTextMLP(nn.Module):
    def __init__(self, inputDimension, outputDimension, isSame = False):
        super().__init__()
        if isSame:
            self.layer1 = nn.Linear(inputDimension, inputDimension)
            self.layer2 = nn.Linear(inputDimension, outputDimension)
        else:
            self.layer1 = nn.Linear(inputDimension, inputDimension//2)
            self.layer2 = nn.Linear(inputDimension//2, outputDimension)
        
        self.gelu = nn.GELU()

    def forward(self, x):
        x = self.layer1(x)
        x = self.gelu(x)
        x = self.layer2(x)
        return x


txt_input = torch.randn(32, 512, 5632)
img_input = torch.randn(32, 64, 768)
concatInpstxt = ConcatenateImgTextMLP(5632, 768)
concatInpsimg = ConcatenateImgTextMLP(768, 768, isSame = True)

txt_input = concatInpstxt(txt_input)
img_input = concatInpsimg(img_input)

modalityEmbeds = nn.Embedding(2, 768)

print(txt_input.shape, img_input.shape)

img_embed = img_input + modalityEmbeds(torch.zeros(64, dtype=torch.long))
txt_embed = txt_input + modalityEmbeds(torch.ones(512, dtype=torch.long))


print(txt_embed.shape, img_embed.shape)
out = torch.concat([img_embed, txt_embed], dim=1)
out.shape

torch.Size([32, 512, 768]) torch.Size([32, 64, 768])
torch.Size([32, 512, 768]) torch.Size([32, 64, 768])


torch.Size([32, 576, 768])

In [10]:
x = torch.randn(2, 3, 512, 512)
dc_ae = AutoencoderDC.from_pretrained("mit-han-lab/dc-ae-f64c128-in-1.0-diffusers", torch_dtype=torch.float32)
with torch.no_grad():
    latents = dc_ae.encode(x).latent
latents.shape
flattened = latents.flatten(2)
flattened.shape

torch.Size([2, 128, 64])

In [27]:
import torch
import torch.nn as nn

class Rotary2DPositionalEncoding(nn.Module):
    def __init__(self, height, width, embedDimension):
        super().__init__()
        self.height = height
        self.width = width
        self.embedDimension = embedDimension

        self.dimHalf = embedDimension // 2
        self.dimQuarter = embedDimension // 4
        inverseFrequency = 1.0 / (500000 ** (torch.arange(0, self.dimQuarter, dtype=torch.float32) / self.dimQuarter))
        # 10000 or 500000 500000 is better from paper, it's just degree by which vector is roated

        heightPositions = torch.arange(height, dtype=torch.float32)
        widthPositions = torch.arange(width, dtype=torch.float32)

        sinusoidHeight = torch.einsum("i,j->ij", heightPositions, inverseFrequency)
        sinusoidWidth = torch.einsum("i,j->ij", widthPositions, inverseFrequency)

        self.register_buffer("sinHeight", sinusoidHeight.sin(), persistent=False)
        self.register_buffer("cosHeight", sinusoidHeight.cos(), persistent=False)
        self.register_buffer("sinWidth", sinusoidWidth.sin(), persistent=False)
        self.register_buffer("cosWidth", sinusoidWidth.cos(), persistent=False)

    def rotateEveryTwo(self, x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)
    
    def applyRope(self, x, sinHeight, cosHeight, sinWidth, cosWidth):

        xHeight = x[..., :self.dimHalf]
        xWidth = x[..., self.dimHalf:]

        sinHeight = sinHeight[None, :, None, :].to(x.device)
        cosHeight = cosHeight[None, :, None, :].to(x.device)
        sinWidth = sinWidth[None, None, :, :].to(x.device)
        cosWidth = cosWidth[None, None, :, :].to(x.device)

        xHeightRotional = (xHeight[..., :self.dimQuarter] * cosHeight) + (self.rotateEveryTwo(xHeight[..., :self.dimQuarter]) * sinHeight)
        xWidthRotional = (xWidth[..., :self.dimQuarter] * cosWidth) + (self.rotateEveryTwo(xWidth[..., :self.dimQuarter]) * sinWidth)

        xHeightRotional = torch.cat([xHeightRotional, xHeight[..., self.dimQuarter:]], dim=-1)
        xWidthRotional = torch.cat([xWidthRotional, xWidth[..., self.dimQuarter:]], dim=-1)
        rotated = torch.cat([xHeightRotional, xWidthRotional], dim=-1)
        return rotated


    def forward(self, x):
        B, L, D = x.shape
        assert D == self.embedDimension
        assert L == self.height * self.width, f"Expected seq_len {self.height*self.width}, got {L}"

        x = x.view(B, self.height, self.width, D)
        x = self.applyRope(x, self.sinHeight, self.cosHeight, self.sinWidth, self.cosWidth)
        return x.view(B, L, D)

rope2D = Rotary2DPositionalEncoding(8, 8, 768)
imagePatches = torch.randn(2, 64, 768)

out = rope2D(imagePatches)
out.shape

torch.Size([2, 64, 768])